# 🏠 House Price Prediction with SAP HANA & Python

**Author:** Your Name  
**Dataset:** California Housing (sklearn)  
**Database:** SAP HANA Cloud  
**Models:** Linear Regression, Random Forest, Gradient Boosting  

## 📌 Project Overview
This project demonstrates a full ML pipeline using SAP HANA Cloud as the data backend:
1. Upload data to SAP HANA Cloud
2. Explore and visualize the dataset
3. Train multiple ML models
4. Compare model performance
5. Save predictions back to HANA

## 📁 Pipeline
```
SAP HANA Cloud
     │
     ▼
Data Loading & EDA
     │
     ▼
Feature Engineering
     │
     ▼
Model Training (3 models)
     │
     ▼
Model Comparison & Evaluation
     │
     ▼
Save Predictions → SAP HANA
```

## ⚙️ Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install hdbcli==2.21.28 hana_ml scikit-learn matplotlib seaborn pandas numpy -q
print('✅ All packages installed!')

## 🔌 Step 2: Connect to SAP HANA Cloud

> **Security:** We use Colab Secrets to hide credentials.  
> Add your password: Click 🔑 (Key icon) → Add secret → Name: `HANA_PASSWORD`

In [ ]:
from hana_ml import dataframe
from google.colab import userdata

# ── HANA Connection Settings ──────────────────────────────
HANA_ADDRESS = "YOUR_HANA_ENDPOINT.hanacloud.ondemand.com"  # ← update this
HANA_PORT    = 443
HANA_USER    = "DBADMIN"
HANA_SCHEMA  = "ML_DEMO"

conn = dataframe.ConnectionContext(
    address=HANA_ADDRESS,
    port=HANA_PORT,
    user=HANA_USER,
    password=userdata.get('HANA_PASSWORD'),  # ✅ secured via Colab Secrets
    encrypt=True,
    sslValidateCertificate=False
)

if conn.connection.isconnected():
    print('✅ Connected to SAP HANA Cloud!')
    print(f'   Version: {conn.hana_version()}')
else:
    print('❌ Connection failed. Check your credentials.')

## 📦 Step 3: Load & Upload Dataset to HANA

In [ ]:
from sklearn.datasets import fetch_california_housing
from hana_ml.dataframe import create_dataframe_from_pandas
import pandas as pd
import numpy as np

# Load dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame
df.columns = [c.replace(' ', '_') for c in df.columns]

print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Columns: {list(df.columns)}')
df.head()

In [ ]:
# Create schema if not exists
cursor = conn.connection.cursor()
try:
    cursor.execute('CREATE SCHEMA ML_DEMO')
    print('✅ Schema ML_DEMO created!')
except Exception as e:
    print(f'ℹ️  Schema note: {e}')
finally:
    cursor.close()

# Upload to HANA
hdf = create_dataframe_from_pandas(
    connection_context=conn,
    pandas_df=df,
    table_name='california_housing',
    schema=HANA_SCHEMA,
    force=True
)
print(f'✅ Data uploaded to HANA! Rows: {hdf.count():,}')

## 📊 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics from HANA
print('=== Dataset Statistics (from SAP HANA) ===')
print(hdf.describe().collect().to_string())

In [ ]:
# Target variable distribution
stats = conn.sql("""
    SELECT
        MIN("MedHouseVal")    AS "Min_Price",
        MAX("MedHouseVal")    AS "Max_Price",
        AVG("MedHouseVal")    AS "Avg_Price",
        STDDEV("MedHouseVal") AS "Std_Price",
        SUM(CASE WHEN "MedHouseVal" > 4.0 THEN 1 ELSE 0 END) AS "Expensive",
        SUM(CASE WHEN "MedHouseVal" BETWEEN 2.0 AND 4.0 THEN 1 ELSE 0 END) AS "Mid_Range",
        SUM(CASE WHEN "MedHouseVal" < 2.0 THEN 1 ELSE 0 END) AS "Affordable"
    FROM "ML_DEMO"."california_housing"
""").collect()

print('=== Target Variable: Median House Value ($100k) ===')
print(stats.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Collect data for visualization
df_plot = hdf.collect()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('California Housing — Exploratory Data Analysis', fontsize=16, fontweight='bold')

# 1. Target distribution
axes[0,0].hist(df_plot['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0,0].set_title('House Price Distribution')
axes[0,0].set_xlabel('Median House Value ($100k)')
axes[0,0].set_ylabel('Count')

# 2. Income vs Price
axes[0,1].scatter(df_plot['MedInc'], df_plot['MedHouseVal'], alpha=0.2, color='coral', s=5)
axes[0,1].set_title('Income vs House Price')
axes[0,1].set_xlabel('Median Income')
axes[0,1].set_ylabel('House Value ($100k)')

# 3. Geographic distribution
sc = axes[0,2].scatter(df_plot['Longitude'], df_plot['Latitude'],
                        c=df_plot['MedHouseVal'], cmap='RdYlGn',
                        alpha=0.3, s=5)
plt.colorbar(sc, ax=axes[0,2], label='House Value ($100k)')
axes[0,2].set_title('Geographic Price Distribution')
axes[0,2].set_xlabel('Longitude')
axes[0,2].set_ylabel('Latitude')

# 4. Price segments
segments = ['Affordable\n(<$200k)', 'Mid-Range\n($200k-$400k)', 'Expensive\n(>$400k)']
counts = [
    len(df_plot[df_plot['MedHouseVal'] < 2.0]),
    len(df_plot[(df_plot['MedHouseVal'] >= 2.0) & (df_plot['MedHouseVal'] <= 4.0)]),
    len(df_plot[df_plot['MedHouseVal'] > 4.0])
]
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[1,0].bar(segments, counts, color=colors, edgecolor='white')
axes[1,0].set_title('Price Segments')
axes[1,0].set_ylabel('Number of Houses')
for i, v in enumerate(counts):
    axes[1,0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

# 5. Correlation heatmap
features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'MedHouseVal']
corr = df_plot[features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1,1], cbar=True, square=True, annot_kws={'size': 8})
axes[1,1].set_title('Feature Correlation Matrix')

# 6. House age distribution
axes[1,2].hist(df_plot['HouseAge'], bins=30, color='mediumpurple', edgecolor='white')
axes[1,2].set_title('House Age Distribution')
axes[1,2].set_xlabel('House Age (years)')
axes[1,2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved as eda_plots.png')

## 🔧 Step 5: Feature Engineering & Train/Test Split

In [ ]:
# Create train/test split using SQL (80/20)
cursor = conn.connection.cursor()

cursor.execute("""
    CREATE OR REPLACE VIEW "ML_DEMO"."housing_train" AS
    SELECT * FROM (
        SELECT ROW_NUMBER() OVER (ORDER BY "MedInc") AS "ID",
               "MedInc", "HouseAge", "AveRooms", "AveBedrms",
               "Population", "AveOccup", "Latitude", "Longitude",
               "MedHouseVal" AS "Target"
        FROM "ML_DEMO"."california_housing"
    ) WHERE MOD("ID", 10) != 0
""")

cursor.execute("""
    CREATE OR REPLACE VIEW "ML_DEMO"."housing_test" AS
    SELECT * FROM (
        SELECT ROW_NUMBER() OVER (ORDER BY "MedInc") AS "ID",
               "MedInc", "HouseAge", "AveRooms", "AveBedrms",
               "Population", "AveOccup", "Latitude", "Longitude",
               "MedHouseVal" AS "Target"
        FROM "ML_DEMO"."california_housing"
    ) WHERE MOD("ID", 10) = 0
""")
cursor.close()

train_hdf = conn.table('housing_train', schema=HANA_SCHEMA)
test_hdf  = conn.table('housing_test',  schema=HANA_SCHEMA)

print(f'✅ Train rows : {train_hdf.count():,}')
print(f'✅ Test rows  : {test_hdf.count():,}')

# Pull to pandas for sklearn
train_pd = train_hdf.collect()
test_pd  = test_hdf.collect()

FEATURES = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
            'Population', 'AveOccup', 'Latitude', 'Longitude']
TARGET   = 'Target'

X_train, y_train = train_pd[FEATURES], train_pd[TARGET]
X_test,  y_test  = test_pd[FEATURES],  test_pd[TARGET]
print(f'✅ Features: {FEATURES}')

## 🤖 Step 6: Train Multiple Models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import time

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05,
        max_depth=5, min_samples_split=10, random_state=42
    )
}

results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    results[name] = {
        'model': model, 'predictions': y_pred,
        'RMSE': rmse, 'MAE': mae, 'R2': r2, 'Time': elapsed
    }
    print(f'✅ RMSE={rmse:.4f} | R²={r2:.4f} | Time={elapsed:.1f}s')

print('\n✅ All models trained!')

## 📈 Step 7: Model Comparison & Evaluation

In [ ]:
# Performance comparison table
comparison = pd.DataFrame([
    {
        'Model': name,
        'RMSE ($100k)': f"{r['RMSE']:.4f}",
        'MAE ($100k)':  f"{r['MAE']:.4f}",
        'R² Score':     f"{r['R2']:.4f}",
        'Train Time':   f"{r['Time']:.1f}s"
    }
    for name, r in results.items()
])

print('=== Model Performance Comparison ===')
print(comparison.to_string(index=False))

best_model_name = max(results, key=lambda x: results[x]['R2'])
print(f'\n🏆 Best Model: {best_model_name} (R²={results[best_model_name]["R2"]:.4f})')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Model Comparison & Evaluation', fontsize=16, fontweight='bold')

model_names  = list(results.keys())
model_colors = ['#3498db', '#2ecc71', '#e74c3c']

# 1. RMSE comparison
rmse_vals = [results[m]['RMSE'] for m in model_names]
bars = axes[0,0].bar(model_names, rmse_vals, color=model_colors, edgecolor='white')
axes[0,0].set_title('RMSE Comparison (lower is better)')
axes[0,0].set_ylabel('RMSE ($100k)')
for bar, val in zip(bars, rmse_vals):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                   f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[0,0].set_xticklabels(model_names, rotation=10)

# 2. R² comparison
r2_vals = [results[m]['R2'] for m in model_names]
bars = axes[0,1].bar(model_names, r2_vals, color=model_colors, edgecolor='white')
axes[0,1].set_title('R² Score Comparison (higher is better)')
axes[0,1].set_ylabel('R² Score')
axes[0,1].set_ylim(0, 1)
for bar, val in zip(bars, r2_vals):
    axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                   f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[0,1].set_xticklabels(model_names, rotation=10)

# 3. MAE comparison
mae_vals = [results[m]['MAE'] for m in model_names]
bars = axes[0,2].bar(model_names, mae_vals, color=model_colors, edgecolor='white')
axes[0,2].set_title('MAE Comparison (lower is better)')
axes[0,2].set_ylabel('MAE ($100k)')
for bar, val in zip(bars, mae_vals):
    axes[0,2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                   f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[0,2].set_xticklabels(model_names, rotation=10)

# 4, 5, 6. Actual vs Predicted for each model
for idx, (name, color) in enumerate(zip(model_names, model_colors)):
    ax = axes[1, idx]
    y_pred = results[name]['predictions']
    ax.scatter(y_test, y_pred, alpha=0.2, color=color, s=5)
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()], 'k--', lw=1.5, label='Perfect')
    ax.set_title(f'{name}\nActual vs Predicted')
    ax.set_xlabel('Actual ($100k)')
    ax.set_ylabel('Predicted ($100k)')
    ax.text(0.05, 0.92, f'R²={results[name]["R2"]:.4f}',
            transform=ax.transAxes, fontsize=11,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.legend()

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison plots saved as model_comparison.png')

## 🔍 Step 8: Feature Importance (Best Model)

In [ ]:
best_model = results[best_model_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Feature': FEATURES,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(importance_df['Feature'], importance_df['Importance'],
                   color='steelblue', edgecolor='white')
    ax.set_xlabel('Feature Importance Score')
    ax.set_title(f'Feature Importances — {best_model_name}', fontsize=14, fontweight='bold')

    for bar, val in zip(bars, importance_df['Importance']):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)

    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n=== Feature Importances ===')
    print(importance_df.sort_values('Importance', ascending=False).to_string(index=False))

## 💾 Step 9: Save Predictions Back to SAP HANA

In [ ]:
# Save best model predictions to HANA
best_preds = results[best_model_name]['predictions']

predictions_df = test_pd[['ID'] + FEATURES + ['Target']].copy()
predictions_df['Predicted']   = best_preds
predictions_df['Error']       = predictions_df['Target'] - predictions_df['Predicted']
predictions_df['Abs_Error']   = predictions_df['Error'].abs()
predictions_df['Model']       = best_model_name

hdf_predictions = create_dataframe_from_pandas(
    connection_context=conn,
    pandas_df=predictions_df,
    table_name='housing_predictions',
    schema=HANA_SCHEMA,
    force=True
)

print(f'✅ Predictions saved to HANA: ML_DEMO.housing_predictions')
print(f'   Rows saved: {hdf_predictions.count():,}')
print(f'   Columns: {hdf_predictions.columns}')

In [ ]:
# Verify from HANA
verification = conn.sql("""
    SELECT
        COUNT(*) AS "Total_Predictions",
        ROUND(AVG(ABS("Error")), 4) AS "Avg_Error",
        ROUND(MIN("Predicted"), 4) AS "Min_Predicted",
        ROUND(MAX("Predicted"), 4) AS "Max_Predicted"
    FROM "ML_DEMO"."housing_predictions"
""")
print('=== Predictions Stored in SAP HANA ===')
print(verification.collect().to_string(index=False))

## 📋 Step 10: Final Summary

In [ ]:
print('=' * 55)
print('   🏠 HOUSE PRICE PREDICTION — PROJECT SUMMARY')
print('=' * 55)
print(f'   Database     : SAP HANA Cloud')
print(f'   Schema       : {HANA_SCHEMA}')
print(f'   Dataset      : California Housing ({len(df):,} rows)')
print(f'   Train / Test : {len(train_pd):,} / {len(test_pd):,}')
print()
print('   Model Results:')
for name, r in results.items():
    marker = '🏆' if name == best_model_name else '  '
    print(f'   {marker} {name:<25} RMSE={r["RMSE"]:.4f}  R²={r["R2"]:.4f}')
print()
print(f'   Best Model   : {best_model_name}')
print(f'   Best RMSE    : {results[best_model_name]["RMSE"]:.4f} ($100k)')
print(f'   Best R²      : {results[best_model_name]["R2"]:.4f}')
print()
print('   Outputs saved to HANA:')
print('   ✅ ML_DEMO.california_housing  (raw data)')
print('   ✅ ML_DEMO.housing_train       (training set)')
print('   ✅ ML_DEMO.housing_test        (test set)')
print('   ✅ ML_DEMO.housing_predictions (model results)')
print('=' * 55)

conn.close()
print('\n✅ Connection closed. Project complete!')